# Person Search – Final Evaluation Pipeline
## YOLO26-Large + PersonViT (ViT-Base, Full FT, Triplet) on PRW

| ID | Student | Email |
|----|---------|-------|
| 0001189110 | Rimondi Simone | simone.rimondi5@studio.unibo.it |

This notebook implements the **end-to-end evaluation pipeline** for the two-stage person search system
trained in the companion notebooks (`Executed_Detector_Training.ipynb` and `Executed_ReIdentificator_Training.ipynb`).

### Pipeline overview
1. **Detection** – YOLO26-Large (`large_full_ft.pt`) localises pedestrians in every gallery frame.
2. **Re-ID** – PersonViT ViT-Base, fine-tuned with Full FT + TripletMarginLoss (`vit_base_full_triplet.pth`),
   embeds each detected crop and every query crop into a 768-dimensional L2-normalised feature space.
3. **Evaluation** – `eval_search_prw` from `eval_function.py` computes **mAP** and **top-1 accuracy**
   following the official PRW person-search protocol.

### Note on `eval_function.py` docstring
The docstring states `gallery_dets` should be `[x1, x2, y1, y2, score]`, but the internal
`_compute_iou` call uses `det[:, :4]` with indices `[0]=x1, [1]=y1, [2]=x2, [3]=y2`, and the
PRW annotation boxes are in `[x1, y1, x2, y2]` format. Therefore the **actual expected format is
`[x1, y1, x2, y2, score]`** – the docstring contains a typo. This notebook uses the correct format.


## 1. Setup & Installations

In [1]:
# Install required libraries
!pip install -q ultralytics
!pip install -q albumentations opencv-python-headless scipy
!pip install -q timm einops yacs pytorch-metric-learning
!pip install -q gdown requests tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 69.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.8/127.8 kB 12.5 MB/s eta 0:00:00


In [2]:
import os, sys, cv2, json, math, time, copy, random, shutil
from pathlib import Path
from collections import defaultdict

import numpy as np
import scipy.io
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm

import albumentations as A
from albumentations.pytorch import ToTensorV2

from ultralytics import YOLO

print("All imports OK")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
All imports OK
Device: cuda


In [3]:
# ── Clone PersonViT (simoswish02 fork with fixed absolute imports) ──────────
if not os.path.exists("PersonViT"):
    os.system("git clone https://github.com/simoswish02/PersonViT")
    print("PersonViT cloned")
else:
    print("PersonViT already present")

sys.path.insert(0, "PersonViT")
from transreid_pytorch.config import cfg as reid_cfg
from transreid_pytorch.model.make_model import make_model
print("TransReID imports OK")

PersonViT cloned
TransReID imports OK


In [4]:
import kagglehub
import shutil

src_dataset = kagglehub.dataset_download('edoardomerli/prw-person-re-identification-in-the-wild')
print('Files saved in:', src_dataset)

# Copy to the main folder
shutil.copytree(src_dataset, '/content/prw', dirs_exist_ok=True)
print('Dataset ready at /content/prw')

100%|██████████| 2.67G/2.67G [02:48<00:00, 17.0MB/s]

Extracting files...


Files saved in: /root/.cache/kagglehub/datasets/edoardomerli/prw-person-re-identification-in-the-wild/versions/1
Dataset ready at /content/prw


In [5]:
# ── Download model weights from GitHub ──────────────────────────────────────
import requests

def download_github_file(url: str, dest: str, desc: str = ""):
    """Download a file from a raw GitHub URL, showing progress."""
    os.makedirs(os.path.dirname(dest) or ".", exist_ok=True)
    if os.path.exists(dest):
        print(f"[skip] {desc or dest} already exists")
        return
    print(f"Downloading {desc or dest} ...")
    r = requests.get(url, stream=True)
    r.raise_for_status()
    total = int(r.headers.get("content-length", 0))
    with open(dest, "wb") as f, tqdm(total=total, unit="B", unit_scale=True, desc=desc) as bar:
        for chunk in r.iter_content(chunk_size=8192):
            f.write(chunk)
            bar.update(len(chunk))
    print(f"  -> saved to {dest}")

# Detector weights (standard GitHub raw URL – not LFS)
DETECTOR_WEIGHTS_URL = (
    "https://github.com/simoswish02/PersonSearch-AssignmentML4CV"
    "/raw/main/weights_detector/large_full_ft.pt"
)
DETECTOR_WEIGHTS_PATH = "weights/large_full_ft.pt"
download_github_file(DETECTOR_WEIGHTS_URL, DETECTOR_WEIGHTS_PATH, "YOLO26-Large weights")

# Re-ID weights (stored in Git LFS – use the LFS media endpoint)
REID_WEIGHTS_URL = (
    "https://media.githubusercontent.com/media/simoswish02/"
    "PersonSearch-AssignmentML4CV/main/weights_reidentificator/vit_base_full_triplet.pth"
)
REID_WEIGHTS_PATH = "weights/vit_base_full_triplet.pth"
download_github_file(REID_WEIGHTS_URL, REID_WEIGHTS_PATH, "ViT-Base Re-ID weights (LFS)")

# eval_function.py
EVAL_FN_URL = (
    "https://raw.githubusercontent.com/simoswish02/"
    "PersonSearch-AssignmentML4CV/main/eval_function.py"
)
download_github_file(EVAL_FN_URL, "eval_function.py", "eval_function.py")

YOLO26-Large weights: 100%|██████████| 52.9M/52.9M [00:00<00:00, 236MB/s]


  -> saved to weights/large_full_ft.pt


ViT-Base Re-ID weights (LFS): 100%|██████████| 346M/346M [00:18<00:00, 18.3MB/s]


  -> saved to weights/vit_base_full_triplet.pth


eval_function.py: 7.34kB [00:00, 16.9MB/s]                   

  -> saved to eval_function.py


## 2. Configuration

In [6]:
class Config:
    # ── Paths ────────────────────────────────────────────────────────────────
    dataset_root          = "/content/prw"
    detector_weights      = DETECTOR_WEIGHTS_PATH
    reid_weights          = REID_WEIGHTS_PATH

    # ── Detector (YOLO26-Large) ───────────────────────────────────────────────
    det_img_size          = 640
    det_conf_thresh       = 0.001   # low threshold: keep all detections, filter in eval_search_prw
    det_iou_nms           = 0.65    # NMS IoU threshold
    det_batch_size        = 8       # images per inference batch

    # ── Re-ID (PersonViT ViT-Base, Full FT, Triplet) ─────────────────────────
    vit_transformer_type  = "vit_base_patch16_224_TransReID"
    vit_embedding_dim     = 768
    img_height            = 256
    img_width             = 128
    reid_batch_size       = 128

    # ── Evaluation ───────────────────────────────────────────────────────────
    det_thresh_eval       = 0.5     # score threshold passed to eval_search_prw
    ignore_cam_id         = True    # ignore camera ID during gallery construction

    # ── Misc ──────────────────────────────────────────────────────────────────
    seed                  = 42
    num_workers           = 4
    min_box_area          = 100     # discard crops smaller than this (pixels²)

# ── Instantiate & seed ──────────────────────────────────────────────────────
cfg = Config()
random.seed(cfg.seed)
np.random.seed(cfg.seed)
torch.manual_seed(cfg.seed)
torch.cuda.manual_seed_all(cfg.seed)

print("Configuration loaded.")
print(f"  Dataset      : {cfg.dataset_root}")
print(f"  Detector     : {cfg.detector_weights}")
print(f"  Re-ID        : {cfg.reid_weights}")
print(f"  Device       : {device}")

Configuration loaded.
  Dataset      : /content/prw
  Detector     : weights/large_full_ft.pt
  Re-ID        : weights/vit_base_full_triplet.pth
  Device       : cuda


## 3. Dataset Definition

We reuse the exact same `PRWPersonSearchDataset` class defined in the Re-ID training notebook.
It supports three modes:
- `train` – annotated training frames (not used at evaluation time, but needed for Re-ID model init)
- `test`  – gallery frames used for detection + feature extraction
- `query` – pre-extracted query crops with ground-truth identity and bounding box


In [16]:
class PRWPersonSearchDataset(Dataset):
    """Loads the PRW dataset in train / test / query mode.

    Annotations are stored as a list of dicts with keys:
        img_name, img_path, boxes (N,4) [x1,y1,x2,y2], pids (N,), cam_id, height, width
    """

    def __init__(self, root, mode="train", min_box_area=100):
        self.root         = root
        self.mode         = mode.lower()
        self.min_box_area = min_box_area
        self.frames_dir   = os.path.join(root, "frames")
        self.ann_dir      = os.path.join(root, "annotations")
        self.annotations  = []

        if self.mode == "query":
            self.query_dir        = os.path.join(root, "query_box")
            self.query_info_file  = os.path.join(root, "query_info.txt")
            self._load_query_data()
        else:
            split_file = os.path.join(root, f"frame_{self.mode}.mat")
            if not os.path.exists(split_file):
                raise FileNotFoundError(f"Split file not found: {split_file}")
            self.ann_files = sorted(self._load_filenames(split_file))
            self._build_annotations()

        print(f"[{self.mode.upper()}] Loaded {len(self.annotations)} samples")

    # ── Internal helpers ─────────────────────────────────────────────────────

    def _load_filenames(self, path):
        mat = scipy.io.loadmat(path)
        key = [k for k in mat.keys() if not k.startswith("__")][0]
        out = []
        for item in mat[key].flatten():
            name = str(item[0]) if isinstance(item, np.ndarray) else str(item)
            ann  = name + ".jpg.mat"
            if os.path.exists(os.path.join(self.ann_dir, ann)):
                out.append(ann)
        return out

    def _extract_camera_id(self, img_name):
        try:
            if img_name.startswith("c"):
                return int(img_name[1])
        except Exception:
            pass
        return 1

    def _build_annotations(self):
        total = ambig = 0
        for af in tqdm(self.ann_files, desc=f"Loading {self.mode}"):
            img_name = af.replace(".mat", "").replace(".jpg", "") + ".jpg"
            img_path = os.path.join(self.frames_dir, img_name)
            if not os.path.exists(img_path):
                continue
            img = cv2.imread(img_path)
            if img is None:
                continue
            H, W  = img.shape[:2]
            vb, vp = [], []
            for box in scipy.io.loadmat(os.path.join(self.ann_dir, af)).get("box_new", []):
                pid = int(box[0])
                x1  = max(0, float(box[1]))
                y1  = max(0, float(box[2]))
                w   = float(box[3])
                h   = float(box[4])
                x2  = min(W, x1 + w)
                y2  = min(H, y1 + h)
                if w > 1 and h > 1 and (x2 - x1) * (y2 - y1) >= self.min_box_area:
                    vb.append([x1, y1, x2, y2])
                    vp.append(pid)
                    total += 1
                    if pid == -2:
                        ambig += 1
            self.annotations.append({
                "img_name": img_name,
                "img_path": img_path,
                "boxes":    np.array(vb, dtype=np.float32) if vb else np.zeros((0, 4), dtype=np.float32),
                "pids":     np.array(vp, dtype=np.int32)   if vp else np.zeros((0,),  dtype=np.int32),
                "cam_id":   self._extract_camera_id(img_name),
                "height":   H,
                "width":    W,
            })
        print(f"  Boxes: {total}  Ambiguous: {ambig}")

    def _load_query_data(self):
        with open(self.query_info_file) as f:
            lines = f.readlines()
        for line in lines:
            parts = line.strip().split()
            if len(parts) != 6:
                continue
            pid = int(parts[0])
            if pid == -2:
                continue
            x, y, w, h = float(parts[1]), float(parts[2]), float(parts[3]), float(parts[4])
            orig_name   = parts[5] + ".jpg"
            crop_name   = f"{pid}_{orig_name}"
            crop_path   = os.path.join(self.query_dir, crop_name)
            if not os.path.exists(crop_path):
                continue
            img = cv2.imread(crop_path)
            if img is None:
                continue
            ch, cw = img.shape[:2]
            self.annotations.append({
                "img_name": crop_name,
                "img_path": crop_path,
                "boxes":    np.array([[x, y, x + w, y + h]], dtype=np.float32),
                "pids":     np.array([pid], dtype=np.int32),
                "cam_id":   self._extract_camera_id(orig_name),
                "height":   ch,
                "width":    cw,
            })

    # ── Dataset protocol ─────────────────────────────────────────────────────

    def __len__(self):
        return len(self.annotations)

    def __getitem__(self, idx):
        ann = self.annotations[idx]
        img = cv2.imread(ann["img_path"])
        if img is None:
            raise ValueError(f"Image not found: {ann['img_path']}")
        return cv2.cvtColor(img, cv2.COLOR_BGR2RGB), ann

    def img_prefix(self):
        """Expose the frames directory path as expected by eval_function.py."""
        return self.frames_dir

    def get_annotation(self, idx):
        return self.annotations[idx]


class QueryDataset(Dataset):
    """Wraps PRWPersonSearchDataset in query mode, applying the test-time transform."""

    def __init__(self, base, transform):
        self.base      = base
        self.transform = transform

    def __len__(self):
        return len(self.base)

    def __getitem__(self, idx):
        img, ann = self.base[idx]
        if self.transform:
            img = self.transform(image=img)["image"]
        return img, ann

In [17]:
# ── Instantiate datasets ────────────────────────────────────────────────────
gallery_dataset = PRWPersonSearchDataset(cfg.dataset_root, mode="test",  min_box_area=cfg.min_box_area)
query_base      = PRWPersonSearchDataset(cfg.dataset_root, mode="query", min_box_area=cfg.min_box_area)

# Test-time transform (same as used during Re-ID training)
test_transform = A.Compose([
    A.Resize(cfg.img_height, cfg.img_width),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2(),
])

query_dataset = QueryDataset(query_base, test_transform)
query_loader  = DataLoader(query_dataset, batch_size=cfg.reid_batch_size,
                           shuffle=False, num_workers=cfg.num_workers, pin_memory=True)

print(f"Gallery frames : {len(gallery_dataset)}")
print(f"Query crops    : {len(query_dataset)}")

Loading test: 100%|██████████| 6112/6112 [00:58<00:00, 103.86it/s]


  Boxes: 24898  Ambiguous: 5902
[TEST] Loaded 6112 samples
[QUERY] Loaded 2057 samples
Gallery frames : 6112
Query crops    : 2057


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


## 4. Evaluation Function

`eval_search_prw` is imported from `eval_function.py` (downloaded from the repository).

**Docstring note** – the docstring lists the detection format as `[x1, x2, y1, y2, score]`,
but the actual implementation (see `_compute_iou` and `det[:, :4]`) assumes **`[x1, y1, x2, y2, score]`**.
Our pipeline produces detections in the correct `[x1, y1, x2, y2, score]` format.


In [9]:
from eval_function import eval_search_prw
print("eval_search_prw imported successfully")

eval_search_prw imported successfully


## 5. Inference Pipeline

### 5a. Detector – YOLO26-Large

Load the fine-tuned YOLO26-Large checkpoint and run inference on every gallery frame.
Results are stored as a list of `np.ndarray` of shape `(N_det, 5)` with columns
`[x1, y1, x2, y2, score]` — matching the format expected by `eval_search_prw`.


In [10]:
# ── Load detector ────────────────────────────────────────────────────────────
detector = YOLO(cfg.detector_weights)
print(f"Detector loaded: {cfg.detector_weights}")

Detector loaded: weights/large_full_ft.pt


In [11]:
def run_detection_on_gallery(detector, gallery_dataset, img_size, conf, iou, batch_size):
    """Run YOLO26-Large on all gallery frames.

    Returns
    -------
    gallery_dets : list[np.ndarray]
        One array per frame, shape (N_det, 5), columns [x1, y1, x2, y2, score].
        Empty frames get a (0, 5) array.
    """
    gallery_dets = []
    img_paths    = [ann["img_path"] for ann in gallery_dataset.annotations]
    n            = len(img_paths)

    for start in tqdm(range(0, n, batch_size), desc="Detection"):
        batch_paths = img_paths[start : start + batch_size]
        results = detector.predict(
            source     = batch_paths,
            imgsz      = img_size,
            conf       = conf,
            iou        = iou,
            device     = device,
            verbose    = False,
            classes    = [0],    # person class only
        )
        for res in results:
            boxes = res.boxes
            if boxes is None or len(boxes) == 0:
                gallery_dets.append(np.zeros((0, 5), dtype=np.float32))
                continue
            xyxy   = boxes.xyxy.cpu().numpy()   # (N, 4)  [x1, y1, x2, y2]
            scores = boxes.conf.cpu().numpy()   # (N,)
            dets   = np.concatenate([xyxy, scores[:, None]], axis=1).astype(np.float32)
            gallery_dets.append(dets)

    return gallery_dets


gallery_dets = run_detection_on_gallery(
    detector      = detector,
    gallery_dataset = gallery_dataset,
    img_size      = cfg.det_img_size,
    conf          = cfg.det_conf_thresh,
    iou           = cfg.det_iou_nms,
    batch_size    = cfg.det_batch_size,
)

total_dets = sum(len(d) for d in gallery_dets)
print(f"Detection complete: {total_dets} total detections across {len(gallery_dets)} frames")
print(f"Average dets/frame: {total_dets / max(len(gallery_dets), 1):.1f}")

Detection: 100%|██████████| 764/764 [04:47<00:00,  2.66it/s]

Detection complete: 55901 total detections across 6112 frames
Average dets/frame: 9.1


### 5b. Re-ID Model – PersonViT ViT-Base (Full FT, Triplet)

Load the fine-tuned ViT-Base checkpoint. The classifier head is replaced with `nn.Identity()`
to expose the raw 768-dim embedding, which is then L2-normalised before cosine similarity computation.


In [12]:
def load_reid_model(weights_path, transformer_type, embedding_dim, reid_cfg_obj, device):
    """Instantiate PersonViT and load fine-tuned weights."""
    mc = copy.deepcopy(reid_cfg_obj)
    mc.MODEL.NAME             = "transformer"
    mc.MODEL.TRANSFORMER_TYPE = transformer_type
    mc.MODEL.JPM              = False
    mc.MODEL.PRETRAIN_CHOICE  = "self"
    mc.MODEL.PRETRAIN_PATH    = weights_path   # placeholder; we load state_dict manually
    mc.MODEL.FEAT_DIM         = embedding_dim
    mc.INPUT.SIZE_TRAIN       = [256, 128]
    mc.INPUT.SIZE_TEST        = [256, 128]
    mc.TEST.NECK_FEAT         = "after"

    # num_class=1 is a dummy value; the head is replaced with Identity
    model = make_model(mc, num_class=1, camera_num=0, view_num=0)
    model.classifier = nn.Identity()

    ckpt = torch.load(weights_path, map_location=device)
    # Support both raw state_dict and wrapped checkpoint formats
    state_dict = ckpt.get("model_state_dict", ckpt)
    model.load_state_dict(state_dict, strict=False)
    model.to(device)
    model.eval()
    return model


reid_model = load_reid_model(
    weights_path     = cfg.reid_weights,
    transformer_type = cfg.vit_transformer_type,
    embedding_dim    = cfg.vit_embedding_dim,
    reid_cfg_obj     = reid_cfg,
    device           = device,
)
total_params = sum(p.numel() for p in reid_model.parameters())
print(f"Re-ID model loaded – {total_params / 1e6:.1f}M parameters")

using Transformer_type: vit_base_patch16_224_TransReID as a backbone
using stride: [16, 16], and patch number is num_y16 * num_x8
Loading pretrained model from weights/vit_base_full_triplet.pth
===========building transformer===========
Re-ID model loaded – 86.5M parameters


### 5c. Gallery Feature Extraction

For each gallery frame, crop the YOLO detections, resize to 256×128, apply ImageNet
normalisation, and extract the 768-dim embedding from the Re-ID model.
Each frame produces one `np.ndarray` of shape `(N_det, 768)`.


In [13]:
@torch.no_grad()
def extract_gallery_features(reid_model, gallery_dataset, gallery_dets,
                              transform, batch_size, device):
    """Extract Re-ID features for every detected crop in the gallery.

    Returns
    -------
    gallery_feats : list[np.ndarray]
        One array per frame, shape (N_det, D). Empty frames get (0, D).
    """
    reid_model.eval()
    D             = cfg.vit_embedding_dim
    gallery_feats = []

    for ann, dets in tqdm(zip(gallery_dataset.annotations, gallery_dets),
                          total=len(gallery_dataset), desc="Gallery feats"):
        if len(dets) == 0:
            gallery_feats.append(np.zeros((0, D), dtype=np.float32))
            continue

        img = cv2.imread(ann["img_path"])
        if img is None:
            gallery_feats.append(np.zeros((0, D), dtype=np.float32))
            continue
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        H, W = img.shape[:2]

        crops = []
        for det in dets:
            x1, y1, x2, y2 = int(det[0]), int(det[1]), int(det[2]), int(det[3])
            x1, y1 = max(0, x1), max(0, y1)
            x2, y2 = min(W, x2), min(H, y2)
            crop = img[y1:y2, x1:x2]
            if crop.size == 0:
                crop = np.zeros((64, 32, 3), dtype=np.uint8)
            crops.append(crop)

        # Process in sub-batches to avoid OOM
        frame_feats = []
        for i in range(0, len(crops), batch_size):
            batch_tensors = torch.stack([
                transform(image=c)["image"] for c in crops[i : i + batch_size]
            ]).to(device)
            out   = reid_model(batch_tensors)
            feats = out[0] if isinstance(out, tuple) else out
            feats = F.normalize(feats, p=2, dim=1)
            frame_feats.append(feats.cpu().numpy())

        gallery_feats.append(np.concatenate(frame_feats, axis=0).astype(np.float32))

    return gallery_feats


gallery_feats = extract_gallery_features(
    reid_model      = reid_model,
    gallery_dataset = gallery_dataset,
    gallery_dets    = gallery_dets,
    transform       = test_transform,
    batch_size      = cfg.reid_batch_size,
    device          = device,
)

total_feats = sum(len(f) for f in gallery_feats)
print(f"Gallery feature extraction complete: {total_feats} feature vectors extracted")

Gallery feats: 100%|██████████| 6112/6112 [08:55<00:00, 11.42it/s]

Gallery feature extraction complete: 55901 feature vectors extracted


### 5d. Query Feature Extraction

Each query is a pre-extracted pedestrian crop. We apply the same test-time transform
and embed it with the Re-ID model to obtain a single 768-dim feature vector per query.


In [14]:
@torch.no_grad()
def extract_query_features(reid_model, query_loader, device):
    """Extract one feature vector per query crop.

    Returns
    -------
    query_box_feats : list[np.ndarray]
        Each element is a (D,) array, one per query.
    """
    reid_model.eval()
    query_box_feats = []

    for imgs, anns in tqdm(query_loader, desc="Query feats"):
        imgs = imgs.to(device)
        out  = reid_model(imgs)
        feats = out[0] if isinstance(out, tuple) else out
        feats = F.normalize(feats, p=2, dim=1).cpu().numpy()
        for feat in feats:
            query_box_feats.append(feat.astype(np.float32))

    return query_box_feats


query_box_feats = extract_query_features(reid_model, query_loader, device)
print(f"Query feature extraction complete: {len(query_box_feats)} query feature vectors")

Query feats:   0%|          | 0/17 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
Query feats: 100%|██████████| 17/17 [00:17<00:00,  1.04s/it]

Query feature extraction complete: 2057 query feature vectors


### 5e. Person Search Evaluation

Call `eval_search_prw` with:
- `gallery_dataset` – the `PRWPersonSearchDataset` in *test* mode
- `query_dataset`   – the `PRWPersonSearchDataset` in *query* mode (we pass `query_base`)
- `gallery_dets`    – list of `(N_det, 5)` arrays `[x1, y1, x2, y2, score]`
- `gallery_feats`   – list of `(N_det, 768)` arrays
- `query_box_feats` – list of `(768,)` arrays
- `det_thresh`      – score threshold to filter low-confidence detections before evaluation


In [18]:
results = eval_search_prw(
    gallery_dataset  = gallery_dataset,
    query_dataset    = query_base,          # PRWPersonSearchDataset in query mode
    gallery_dets     = gallery_dets,
    gallery_feats    = gallery_feats,
    query_box_feats  = query_box_feats,
    det_thresh       = cfg.det_thresh_eval,
    ignore_cam_id    = cfg.ignore_cam_id,
)

print("=" * 50)
print("FINAL EVALUATION RESULTS")
print("=" * 50)
print(f"  mAP   : {results['mAP'] * 100:.2f}%")
print(f"  Top-1 : {results['accs'][0] * 100:.2f}%")
print("=" * 50)

search ranking:
  mAP = 79.10%
  top- 1 = 97.18%
FINAL EVALUATION RESULTS
  mAP   : 79.10%
  Top-1 : 97.18%
